# STEP 1: OpenDART API 직접 호출

이 노트북의 목표는 다음 조회 순서를 직접 확인하는 것입니다.

`기업명 → corp_code → 전체 재무제표 raw JSON → 주요 재무정보`

현재는 학습을 위해 Python 표준 라이브러리만 사용합니다. LangChain, LangGraph, LlamaIndex, LLM, Agent는 사용하지 않습니다.

> 현재 하위 단계에서는 노트북만 작성합니다. 여기서 동작을 확인한 코드는 이후 요청에서 `src/dart/client.py`로 옮기고, 이 노트북은 해당 모듈을 import하도록 바꿀 예정입니다.

## 0. 실행 전 준비

프로젝트 루트에 `.env` 파일을 만들고 아래처럼 API 키를 입력합니다.

```text
OPENDART_API_KEY=발급받은_API_KEY
```

API 키는 출력하거나 Git에 커밋하지 마세요.

## 1. 표준 라이브러리 불러오기

- `urllib`: HTTP GET 요청과 쿼리 문자열 생성
- `zipfile`, `io`: 기업 고유번호 ZIP 파일을 메모리에서 열기
- `xml.etree.ElementTree`: 기업 목록 XML 읽기
- `json`: 재무제표 JSON 읽기

In [3]:
import io
import json
import os
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path

## 2. 프로젝트 루트와 `.env` 위치 확인

Jupyter를 프로젝트 루트에서 실행해도, `notebooks/`에서 실행해도 같은 `.env`를 찾도록 합니다.

In [4]:
current_dir = Path.cwd().resolve()
if current_dir.name == "learning":
    project_root = current_dir.parents[1]
elif current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir
env_path = project_root / ".env"

print("프로젝트 루트:", project_root)
print(".env 존재 여부:", env_path.exists())

프로젝트 루트: 확인됨
.env 존재 여부: True


## 3. `.env`에서 API 키 읽기

이번 단계에서는 별도 패키지를 추가하지 않고 `KEY=VALUE` 형식의 최소 `.env`만 읽습니다.

In [5]:
if not env_path.exists():
    raise FileNotFoundError(f"{env_path} 파일을 먼저 만들어 주세요.")

for raw_line in env_path.read_text(encoding="utf-8").splitlines():
    line = raw_line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    key, value = line.split("=", 1)
    os.environ.setdefault(key.strip(), value.strip().strip("'\""))

In [6]:
api_key = os.environ.get("OPENDART_API_KEY")

if not api_key:
    raise ValueError(".env에 OPENDART_API_KEY를 설정해 주세요.")

print(f"API 키를 읽었습니다. (길이: {len(api_key)}자, 값은 출력하지 않음)")

API 키를 읽었습니다. (길이: 40자, 값은 출력하지 않음)


## 4. 조회 조건 정하기

다른 기업이나 연도를 조회하려면 아래 두 값만 바꿉니다. 사업보고서(`11011`)와 연결재무제표(`CFS`)를 조회합니다.

In [7]:
corp_name = "삼성전자"
business_year = "2025"
report_code = "11011"  # 사업보고서
fs_div = "CFS"         # 연결재무제표

## 5. OpenDART URL 준비

기업 고유번호와 전체 재무제표는 서로 다른 API에서 가져옵니다.

In [8]:
corp_code_url = "https://opendart.fss.or.kr/api/corpCode.xml"
financials_url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

## 6. 작은 HTTP GET 함수 만들기

요청 파라미터를 URL 쿼리 문자열로 바꾸고 응답 본문을 `bytes`로 받습니다. 응답 형식이 ZIP과 JSON으로 다르므로 여기서는 아직 해석하지 않습니다. HTTP 오류에서는 API 키가 포함된 요청 URL을 출력하지 않습니다.

In [9]:
def get_bytes(url, params, timeout=30):
    query = urllib.parse.urlencode(params)
    request_url = f"{url}?{query}"

    try:
        with urllib.request.urlopen(request_url, timeout=timeout) as response:
            return response.read()
    except urllib.error.HTTPError as error:
        raise RuntimeError(f"OpenDART HTTP 오류: {error.code}") from None
    except urllib.error.URLError as error:
        raise RuntimeError(f"네트워크 연결 오류: {error.reason}") from None

## 7. 전체 기업 고유번호 파일 요청

OpenDART는 기업명을 바로 입력받지 않습니다. 먼저 전체 기업 목록 ZIP을 받은 뒤, 그 안에서 기업명을 검색해야 합니다.

In [10]:
corp_zip_bytes = get_bytes(
    corp_code_url,
    {"crtfc_key": api_key},
)

print(f"받은 데이터 크기: {len(corp_zip_bytes):,} bytes")

받은 데이터 크기: 3,600,217 bytes


## 8. ZIP 응답인지 확인

키가 잘못되면 정상 ZIP 대신 오류 XML이 올 수 있습니다. ZIP이 아니면 응답 내용을 먼저 확인합니다.

In [11]:
corp_zip_stream = io.BytesIO(corp_zip_bytes)

if not zipfile.is_zipfile(corp_zip_stream):
    error_text = corp_zip_bytes.decode("utf-8", errors="replace")
    raise RuntimeError(f"기업 고유번호 응답이 ZIP이 아닙니다:\n{error_text}")

print("정상 ZIP 응답입니다.")

정상 ZIP 응답입니다.


## 9. ZIP 안의 XML 꺼내기

파일을 디스크에 저장하지 않고 메모리에서 바로 읽습니다.

In [12]:
with zipfile.ZipFile(corp_zip_stream) as archive:
    file_names = archive.namelist()
    print("ZIP 내부 파일:", file_names)
    corp_xml_bytes = archive.read(file_names[0])

ZIP 내부 파일: ['CORPCODE.xml']


## 10. XML을 Python 데이터로 변환

각 `<list>` 요소가 기업 하나를 뜻합니다. `corp_code`, `corp_name`, `stock_code`, `modify_date` 등을 가진 딕셔너리 목록으로 바꿉니다.

In [13]:
corp_root = ET.fromstring(corp_xml_bytes)
corp_records = [
    {child.tag: child.text for child in item}
    for item in corp_root.findall("list")
]

print(f"기업 수: {len(corp_records):,}")
corp_records[:3]

기업 수: 118,819


[{'corp_code': '00434003',
  'corp_name': '다코',
  'corp_eng_name': 'Daco corporation',
  'stock_code': ' ',
  'modify_date': '20170630'},
 {'corp_code': '00430964',
  'corp_name': '굿앤엘에스',
  'corp_eng_name': 'Good & LS Co.,Ltd.',
  'stock_code': ' ',
  'modify_date': '20170630'},
 {'corp_code': '00388953',
  'corp_name': '크레디피아제이십오차유동화전문회사',
  'corp_eng_name': 'Credipia 25th Asset Securitization Specialty L.L.C',
  'stock_code': ' ',
  'modify_date': '20170630'}]

## 11. 기업명으로 `corp_code` 찾기

현재는 학습을 위해 정식 회사명과 정확히 일치하는 기업을 찾습니다.

In [14]:
matches = [
    record
    for record in corp_records
    if record.get("corp_name") == corp_name
]

matches

[{'corp_code': '00126380',
  'corp_name': '삼성전자',
  'corp_eng_name': 'SAMSUNG ELECTRONICS CO,.LTD',
  'stock_code': '005930',
  'modify_date': '20251201'}]

In [15]:
if len(matches) != 1:
    raise LookupError(
        f"{corp_name!r}과 정확히 일치하는 기업이 1개가 아닙니다: {len(matches)}개"
    )

corp_info = matches[0]
corp_code = corp_info["corp_code"]

print("기업명:", corp_info["corp_name"])
print("corp_code:", corp_code)
print("종목코드:", corp_info.get("stock_code"))

기업명: 삼성전자
corp_code: 00126380
종목코드: 005930


## 12. 전체 재무제표 API 요청

`11011`은 사업보고서, `CFS`는 연결재무제표를 뜻합니다.

In [16]:
financial_params = {
    "crtfc_key": api_key,
    "corp_code": corp_code,
    "bsns_year": business_year,
    "reprt_code": report_code,
    "fs_div": fs_div,
}

In [17]:
financial_json_bytes = get_bytes(financials_url, financial_params)
raw_response = json.loads(financial_json_bytes.decode("utf-8"))

print(f"받은 데이터 크기: {len(financial_json_bytes):,} bytes")

받은 데이터 크기: 114,947 bytes


## 13. raw JSON 전체 확인

아래 셀은 가공하지 않은 응답 전체를 표시합니다. 먼저 최상위의 `status`, `message`, `list` 구조를 확인하세요.

In [18]:
raw_response

{'status': '000',
 'message': '정상',
 'list': [{'rcept_no': '20260310002820',
   'reprt_code': '11011',
   'bsns_year': '2025',
   'corp_code': '00126380',
   'sj_div': 'BS',
   'sj_nm': '재무상태표',
   'account_id': 'ifrs-full_Assets',
   'account_nm': '자산총계',
   'account_detail': '-',
   'thstrm_nm': '제 57 기',
   'thstrm_amount': '566942110000000',
   'frmtrm_nm': '제 56 기',
   'frmtrm_amount': '514531948000000',
   'bfefrmtrm_nm': '제 55 기',
   'bfefrmtrm_amount': '455905980000000',
   'ord': '7',
   'currency': 'KRW'},
  {'rcept_no': '20260310002820',
   'reprt_code': '11011',
   'bsns_year': '2025',
   'corp_code': '00126380',
   'sj_div': 'BS',
   'sj_nm': '재무상태표',
   'account_id': 'ifrs-full_CurrentAssets',
   'account_nm': '유동자산',
   'account_detail': '-',
   'thstrm_nm': '제 57 기',
   'thstrm_amount': '247684612000000',
   'frmtrm_nm': '제 56 기',
   'frmtrm_amount': '227062266000000',
   'bfefrmtrm_nm': '제 55 기',
   'bfefrmtrm_amount': '195936557000000',
   'ord': '8',
   'currency': '

## 14. 응답 상태와 한 개 계정 확인

정상 상태 코드는 `000`입니다. 각 `list` 원소는 재무제표의 한 계정과목입니다.

In [19]:
status = raw_response.get("status")
message = raw_response.get("message")

print("status:", status)
print("message:", message)

if status != "000":
    raise RuntimeError(f"OpenDART API 오류: [{status}] {message}")

status: 000
message: 정상


In [20]:
account_rows = raw_response["list"]

print(f"계정과목 수: {len(account_rows):,}")
account_rows[0]

계정과목 수: 229


{'rcept_no': '20260310002820',
 'reprt_code': '11011',
 'bsns_year': '2025',
 'corp_code': '00126380',
 'sj_div': 'BS',
 'sj_nm': '재무상태표',
 'account_id': 'ifrs-full_Assets',
 'account_nm': '자산총계',
 'account_detail': '-',
 'thstrm_nm': '제 57 기',
 'thstrm_amount': '566942110000000',
 'frmtrm_nm': '제 56 기',
 'frmtrm_amount': '514531948000000',
 'bfefrmtrm_nm': '제 55 기',
 'bfefrmtrm_amount': '455905980000000',
 'ord': '7',
 'currency': 'KRW'}

직접 확인할 주요 필드는 다음과 같습니다.

- `account_id`: XBRL 계정 식별자
- `account_nm`: 사람이 읽는 계정명
- `sj_div`, `sj_nm`: 재무제표 구분과 이름
- `thstrm_nm`, `thstrm_amount`: 당기 이름과 금액
- `currency`: 통화 단위
- `rcept_no`: 해당 공시의 접수번호
- `fs_div`: 연결(`CFS`) 또는 별도(`OFS`) 재무제표 구분

## 15. 필요한 세 계정만 찾기

회사에 따라 표시 계정명이 달라질 수 있으므로 먼저 XBRL `account_id`를 사용하고, 학습 예제에 필요한 최소한의 계정명 후보를 보조로 사용합니다.

In [21]:
target_accounts = {
    "매출액": {
        "ids": {"ifrs-full_Revenue", "ifrs-full_RevenueFromContractsWithCustomers"},
        "names": {"매출액", "수익(매출액)"},
    },
    "영업이익": {
        "ids": {"dart_OperatingIncomeLoss"},
        "names": {"영업이익", "영업이익(손실)"},
    },
    "당기순이익": {
        "ids": {"ifrs-full_ProfitLoss"},
        "names": {"당기순이익", "당기순이익(손실)"},
    },
}

In [22]:
def find_account(rows, target):
    income_rows = [row for row in rows if row.get("sj_div") in {"IS", "CIS"}]

    by_id = next(
        (row for row in income_rows if row.get("account_id") in target["ids"]),
        None,
    )
    if by_id:
        return by_id

    return next(
        (row for row in income_rows if row.get("account_nm") in target["names"]),
        None,
    )

In [23]:
selected_accounts = {
    label: find_account(account_rows, target)
    for label, target in target_accounts.items()
}

selected_accounts

{'매출액': {'rcept_no': '20260310002820',
  'reprt_code': '11011',
  'bsns_year': '2025',
  'corp_code': '00126380',
  'sj_div': 'IS',
  'sj_nm': '손익계산서',
  'account_id': 'ifrs-full_Revenue',
  'account_nm': '매출액',
  'account_detail': '-',
  'thstrm_nm': '제 57 기',
  'thstrm_amount': '333605938000000',
  'thstrm_add_amount': '',
  'frmtrm_nm': '제 56 기',
  'frmtrm_amount': '300870903000000',
  'bfefrmtrm_nm': '제 55 기',
  'bfefrmtrm_amount': '258935494000000',
  'ord': '23',
  'currency': 'KRW'},
 '영업이익': {'rcept_no': '20260310002820',
  'reprt_code': '11011',
  'bsns_year': '2025',
  'corp_code': '00126380',
  'sj_div': 'IS',
  'sj_nm': '손익계산서',
  'account_id': 'dart_OperatingIncomeLoss',
  'account_nm': '영업이익',
  'account_detail': '-',
  'thstrm_nm': '제 57 기',
  'thstrm_amount': '43601051000000',
  'thstrm_add_amount': '',
  'frmtrm_nm': '제 56 기',
  'frmtrm_amount': '32725961000000',
  'bfefrmtrm_nm': '제 55 기',
  'bfefrmtrm_amount': '6566976000000',
  'ord': '6',
  'currency': 'KRW'},
 '당기

## 16. 주요 재무정보로 정리

금액은 OpenDART 원문 값을 유지합니다. 찾지 못한 계정은 `None`이므로 바로 앞 셀과 raw JSON에서 실제 계정명을 확인하세요.

In [24]:
summary = {
    "기업명": corp_info["corp_name"],
    "사업연도": business_year,
    **{
        label: row.get("thstrm_amount") if row else None
        for label, row in selected_accounts.items()
    },
}

summary

{'기업명': '삼성전자',
 '사업연도': '2025',
 '매출액': '333605938000000',
 '영업이익': '43601051000000',
 '당기순이익': '45206805000000'}

## 이번 노트북에서 이해할 핵심

1. 기업명은 재무제표 API의 입력값이 아니므로 먼저 `corp_code`로 변환해야 합니다.
2. HTTP 응답은 ZIP, XML, JSON처럼 서로 다른 형식일 수 있습니다.
3. JSON의 `status == '000'`인지 확인한 뒤 `list`를 사용해야 합니다.
4. 사업연도만으로는 부족하며 보고서 종류와 연결/별도 재무제표도 선택해야 합니다.
5. 재무 계정은 표시 이름이 달라질 수 있어 raw JSON의 `account_id`, `account_nm`, `sj_div`를 함께 확인해야 합니다.

여기까지가 현재 하위 단계입니다. 아직 `src/`, 테스트 또는 Agent 구조를 만들지 않습니다.